# Name That LLM — offline CPU inference entry

This Notebook fits the frozen TF-IDF classifier on the official training data,
then writes one `id, model` prediction per actual test row to
`/kaggle/working/submission.parquet`. It does not download external models,
packages, or data. Use Accelerator **None** and Internet **Off**, and attach the
official Name That LLM competition data. Saving or running this Notebook does
not constitute a competition submission.

The configuration was chosen using local prompt-grouped development validation.
The public test example only checks the interface. This Notebook has not been
executed on Kaggle. Runtime package versions are logged; exact reruns require
the same versions as the local experiment (see the version cell). If Kaggle's
preinstalled packages differ, attach compatible offline wheels or treat the
run as a separately versioned reproduction and record its results.


In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '2')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '2')
import importlib.metadata
import json
import platform
import time
from pathlib import Path
from threadpoolctl import threadpool_limits

INPUT = Path(os.environ.get('NTL_INPUT_DIR', '/kaggle/input/competitions/name-that-llm'))
OUTPUT = Path(os.environ.get('NTL_OUTPUT_DIR', '/kaggle/working'))
OUTPUT.mkdir(parents=True, exist_ok=True)
EXPECTED_VERSIONS = {'scikit-learn': '1.9.1', 'numpy': '2.5.4', 'pyarrow': '26.0.0', 'scipy': '1.18.1', 'joblib': '1.6.0'}
EXPECTED_PYTHON = '3.13.2'
ACTUAL_VERSIONS = {key: importlib.metadata.version(key) for key in EXPECTED_VERSIONS}
VERSION_MISMATCH = {key: {'local': value, 'runtime': ACTUAL_VERSIONS[key]}
                    for key, value in EXPECTED_VERSIONS.items()
                    if ACTUAL_VERSIONS[key] != value}
if platform.python_version() != EXPECTED_PYTHON:
    VERSION_MISMATCH['python'] = {'local': EXPECTED_PYTHON, 'runtime': platform.python_version()}
print({'python': platform.python_version(), 'packages': ACTUAL_VERSIONS,
       'version_mismatches': VERSION_MISMATCH})


In [ ]:
"""Reusable, offline TF-IDF classification and strict submission checks."""
from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC


def read_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def config_id(config):
    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]


def feature_config(config):
    return {key: value for key, value in config.items()
            if key not in {"classifier", "C", "class_weight"}}


def make_features(config):
    common = dict(lowercase=config.get("lowercase", False),
                  sublinear_tf=config.get("sublinear_tf", True),
                  min_df=config.get("min_df", 2), dtype=np.float32, norm="l2")
    representations = []
    if config["representation"] in {"char", "combined"}:
        representations.append(("char", TfidfVectorizer(
            analyzer="char", ngram_range=tuple(config.get("char_ngram", [3, 5])),
            max_features=100000, **common)))
    if config["representation"] in {"word", "combined"}:
        representations.append(("word", TfidfVectorizer(
            analyzer="word", ngram_range=tuple(config.get("word_ngram", [1, 2])),
            token_pattern=r"(?u)\b\w+\b", max_features=50000, **common)))
    if not representations:
        raise ValueError("Unknown representation")
    return FeatureUnion(representations, n_jobs=1)


def make_classifier(config, seed=20261018):
    common = dict(C=config.get("C", 1.0), class_weight=config.get("class_weight"),
                  random_state=seed, tol=1e-4)
    if config.get("classifier", "svm") == "svm":
        return LinearSVC(dual=True, max_iter=10000, **common)
    if config["classifier"] == "logistic":
        return LogisticRegression(solver="lbfgs", max_iter=1000, **common)
    raise ValueError("Unknown classifier")


def make_pipeline(config, seed=20261018):
    return Pipeline([("features", make_features(config)),
                     ("classifier", make_classifier(config, seed))])


def checked_submission(test_rows, predictions, labels, output):
    """Preserve every test ID and actual row count; never infer hidden size."""
    import pyarrow as pa
    import pyarrow.parquet as pq
    ids = [row["id"] for row in test_rows]
    predictions = [str(value) for value in predictions]
    if not ids or any(not str(value).strip() for value in ids) or len(set(ids)) != len(ids):
        raise ValueError("Test IDs must be nonblank and unique")
    if len(predictions) != len(ids) or any(value not in labels for value in predictions):
        raise ValueError("Predictions must cover every test row with a reference label")
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    pq.write_table(pa.table({"id": ids, "model": predictions}), output)
    result = pq.read_table(output).to_pydict()
    if result["id"] != ids or result["model"] != predictions:
        raise RuntimeError("Parquet round-trip validation failed")
    return {"rows": len(ids), "unique_ids": len(set(ids)), "labels_valid": True,
            "sha256": file_hash(output)}


In [ ]:
"""Second CPU round: interpretable style features and independent score fusion."""
from __future__ import annotations

import re
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler, Normalizer

import types
original = types.SimpleNamespace(make_features=make_features, make_classifier=make_classifier)


class StylometryFeatures(TransformerMixin, BaseEstimator):
    """Text-only counts and rates. No IDs, prompts, labels or external resources."""
    def fit(self, texts, y=None):
        return self

    def transform(self, texts):
        rows = []
        for text in texts:
            words = re.findall(r"\w+", text, flags=re.UNICODE)
            lengths = np.array([len(word) for word in words], dtype=float)
            n, count = max(len(text), 1), max(len(words), 1)
            sentence_count = max(len(re.findall(r"[.!?]+", text)), 1)
            values = [np.log1p(len(text)), np.log1p(len(words)), np.log1p(text.count("\n")),
                      float(lengths.mean()) if len(lengths) else 0.,
                      float(lengths.std()) if len(lengths) else 0.,
                      len(set(word.lower() for word in words)) / count,
                      count / sentence_count, sum(char.isupper() for char in text) / n,
                      sum(char.isdigit() for char in text) / n,
                      sum(char.isspace() for char in text) / n,
                      sum(ord(char) > 127 for char in text) / n]
            values.extend(text.count(char) / n for char in
                          ("\n", ",", ";", ":", "!", "?", "(", "[", "-", "—", "'", '"', "`", "*", "/"))
            rows.append(values)
        return np.asarray(rows, dtype=np.float32)


def make_features(config):
    features = original.make_features(config)
    if config.get("char_analyzer") == "char_wb":
        features.set_params(char__analyzer="char_wb")
    if config.get("style_weight", 0):
        style = Pipeline([("counts", StylometryFeatures()), ("scale", StandardScaler()),
                          ("normalize", Normalizer())])
        features = FeatureUnion([("text", features), ("style", style)], n_jobs=1,
                                transformer_weights={"text": 1., "style": config["style_weight"]})
    return features


def normalize_scores(scores):
    return scores / np.maximum(np.linalg.norm(scores, axis=1, keepdims=True), 1e-12)


class ScoreFusion(ClassifierMixin, BaseEstimator):
    def __init__(self, char_config, word_config, char_weight=.5):
        self.char_config = char_config
        self.word_config = word_config
        self.char_weight = char_weight

    def fit(self, texts, y):
        self.char_model_ = make_pipeline(self.char_config)
        self.word_model_ = make_pipeline(self.word_config)
        self.char_model_.fit(texts, y)
        self.word_model_.fit(texts, y)
        self.classes_ = self.char_model_.classes_
        if not np.array_equal(self.classes_, self.word_model_.classes_):
            raise ValueError("Fusion component classes differ")
        return self

    def decision_function(self, texts):
        return (self.char_weight * normalize_scores(self.char_model_.decision_function(texts))
                + (1 - self.char_weight) * normalize_scores(self.word_model_.decision_function(texts)))

    def predict(self, texts):
        return self.classes_[self.decision_function(texts).argmax(axis=1)]


def make_pipeline(config, seed=20261018):
    if config.get("classifier") == "score_fusion":
        return ScoreFusion(config["char_config"], config["word_config"], config["char_weight"])
    return Pipeline([("features", make_features(config)),
                     ("classifier", original.make_classifier(config, seed))])


In [ ]:
CONFIG = {'classifier': 'logistic', 'C': 16.0, 'class_weight': None, 'lowercase': False, 'sublinear_tf': True, 'min_df': 2, 'char_ngram': [2, 5], 'word_ngram': [1, 2], 'representation': 'char', 'style_weight': 0.25}
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
EXPECTED_TRAIN_SHA256 = '40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707'

train_path = INPUT / 'train.csv'
test_path = INPUT / 'test.csv'
if file_hash(train_path) != EXPECTED_TRAIN_SHA256:
    raise ValueError('Training data differs from the audited snapshot; re-audit before fitting.')
train = read_csv(train_path)
test = read_csv(test_path)
if not train or not test:
    raise ValueError('Official train and test files must contain rows.')
if set(row['model'] for row in train) != set(LABELS):
    raise ValueError('Actual training labels differ from the frozen reference.')
if any(not row['text'].strip() for row in train + test):
    raise ValueError('Blank text requires a new data audit.')
started = time.monotonic()
with threadpool_limits(limits=2):
    pipeline = make_pipeline(CONFIG)
    pipeline.fit([row['text'] for row in train], [LABELS.index(row['model']) for row in train])
    predictions = [LABELS[index] for index in pipeline.predict([row['text'] for row in test])]
checks = checked_submission(test, predictions, LABELS, OUTPUT / 'submission.parquet')
runtime = {'config': CONFIG, 'train_sha256': EXPECTED_TRAIN_SHA256,
           'train_rows': len(train), 'test_rows': len(test),
           'elapsed_seconds': time.monotonic() - started,
           'python': platform.python_version(),
           'full_training_classifier_fits': 2 if CONFIG.get('classifier') == 'score_fusion' else 1,
           'versions': ACTUAL_VERSIONS, 'version_mismatches': VERSION_MISMATCH,
           'output_validation': checks, 'formal_submission': False}
(OUTPUT / 'runtime.json').write_text(json.dumps(runtime, indent=2) + '\n')
print(runtime)
